# Proyecto de Grado — Simulador de Escenarios Comerciales para el Sector Apícola
## Componente de Inteligencia Artificial y Series Temporales

Este Notebook documenta y ejecuta el flujo completo de ingeniería de datos, modelado predictivo y lógica de negocio para la gestión de inventarios utilizando el modelo de Machine Learning **Meta Prophet**.

**Universidad Mayor, Real y Pontificia de San Francisco Xavier de Chuquisaca**  
Ingeniería en Tecnologías de la Información y Seguridad  

---

### Objetivos del Notebook

1. **Análisis Histórico**: Carga y preparación de datos transaccionales simulados.
2. **Entrenamiento Predictivo**: Configuración del modelo Prophet con estacionalidades anuales y semanales.
3. **Simulación de Factores Externos**: Inyección de variables exógenas (floración de primavera, competencia de mercado).
4. **Lógica de Reabastecimiento Automatizado**: Cálculo del punto de reorden para mitigar quiebres de stock.

## Paso 0 — Instalación de Dependencias

Ejecuta esta celda **una sola vez** si estás en Google Colab o en un entorno sin Prophet instalado. En un entorno local con el paquete ya disponible puedes omitirla.

In [ ]:
# Instalar Prophet (solo necesario en entornos nuevos como Google Colab)
# Descomenta la siguiente línea si Prophet no está instalado:
# !pip install prophet --quiet

print('Entorno listo. Procede con las celdas siguientes.')

## Paso 1 — Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
import matplotlib.pyplot as plt
import pickle
import os

print('✅ Librerías cargadas exitosamente. Meta Prophet listo para el análisis.')

## Paso 2 — Catálogo de Productos y Generación de Historial Simulado

Para la demostración con el sector apícola en Sucre definimos el catálogo con los 14 productos clave (cera estampada, suplementos de colmenas, tratamientos para Varroa, etc.).

Generamos un DataFrame histórico de **2 años** (2024-2026) para el producto estrella: **SINCROCIP — Cera Estampada de Abeja**.

La serie incluye:
- Una tendencia base con pico los viernes (día de feria en Sucre).
- Estacionalidad anual que simula la temporada de floración (pico en primavera).
- Ruido gaussiano para modelar la variabilidad real del mercado.

In [ ]:
# Rango de fechas: 2 años de historial
dates = pd.date_range(start='2024-01-01', end='2026-06-01', freq='D')
np.random.seed(42)

# Demanda base + efecto viernes + estacionalidad anual + ruido
base_sales      = 15 + (dates.dayofweek == 4) * 5
seasonal_effect = np.sin(2 * np.pi * dates.dayofyear / 365.25) * 8
noise           = np.random.normal(0, 3, len(dates))
sales_volume    = np.maximum(5, base_sales + seasonal_effect + noise).astype(int)

# DataFrame en el formato estricto que exige Prophet: columnas 'ds' e 'y'
df_historico = pd.DataFrame({'ds': dates, 'y': sales_volume})

print(f'📈 Datos históricos generados para SINCROCIP.')
print(f'   Total de registros: {len(df_historico)}')
print(df_historico.head(10).to_string(index=False))

## Paso 3 — Visualización del Historial de Ventas

Antes de entrenar el modelo es buena práctica inspeccionar visualmente la serie para confirmar que la estacionalidad y la tendencia son coherentes con el negocio.

In [ ]:
plt.figure(figsize=(13, 4))
plt.plot(df_historico['ds'], df_historico['y'], color='#2563eb', linewidth=0.9, alpha=0.8)
plt.title('Historial de Ventas — SINCROCIP (Cera Estampada de Abeja)', fontsize=13, fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('Unidades vendidas')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

## Paso 4 — Entrenamiento del Modelo Meta Prophet

Prophet descompone la serie temporal en tres componentes mediante un modelo aditivo generalizado:

```
y(t) = tendencia(t) + estacionalidad(t) + efectos_especiales(t) + error(t)
```

Configuración utilizada:
- `yearly_seasonality=True` — captura el ciclo anual de floración.
- `weekly_seasonality=True` — captura el pico de los días de feria.
- `daily_seasonality=False` — datos diarios no requieren subestacionalidad intradía.

El modelo entrenado se serializa en un archivo binario `.pkl` tal como lo realiza el **Backend Predictor** del sistema en producción (puerto 8001).

In [ ]:
# Inicializar y entrenar
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)
model.fit(df_historico)

# Serializar el modelo entrenado
modelo_path = 'modelo_SINCROCIP.pkl'
with open(modelo_path, 'wb') as f:
    pickle.dump(model, f)

print(f'💾 Modelo Prophet entrenado y exportado en: "{modelo_path}"')
print(f'   Tamaño del archivo: {os.path.getsize(modelo_path) / 1024:.1f} KB')

## Paso 5 — Motor de Simulación: Factores Exógenos

En este paso emulamos la petición que el **Frontend** envía al Backend Predictor cuando el usuario configura un escenario en la interfaz.

**Configuración del escenario de ejemplo:**

| Parámetro | Valor | Justificación |
|---|---|---|
| Periodo de proyección | 30 días | Horizonte operativo mensual |
| Factor Estacional (Primavera/Floración) | +25 % | Alta actividad apícola en Chuquisaca |
| Factor Competencia (Nuevos competidores) | −10 % | Presión de mercado estimada |

El ajuste se aplica como **multiplicador** sobre la predicción inercial del modelo.

In [ ]:
# 1. Cargar el modelo desde el binario
with open('modelo_SINCROCIP.pkl', 'rb') as f:
    model_loaded = pickle.load(f)

# 2. Proyectar los próximos 30 días
future   = model_loaded.make_future_dataframe(periods=30, freq='D')
forecast = model_loaded.predict(future)

# 3. Aislar únicamente el horizonte proyectado
prediccion_futura = forecast.tail(30).copy()

# 4. Aplicar factores exógenos seleccionados por el usuario en la UI
factor_estacional   = 1.25   # +25 % por floración de primavera
factor_competencia  = 0.90   # −10 % por competencia de mercado

prediccion_futura['yhat_ajustado'] = (
    prediccion_futura['yhat'] * factor_estacional * factor_competencia
)

print('🔮 Predicciones base y ajustadas (primeros 10 días del horizonte):')
print(
    prediccion_futura[['ds', 'yhat', 'yhat_ajustado']]
    .head(10)
    .rename(columns={'ds': 'Fecha', 'yhat': 'Base', 'yhat_ajustado': 'Ajustada'})
    .to_string(index=False)
)

## Paso 6 — Lógica de Alertas Logísticas y Punto de Reorden

El sistema cruza la predicción ajustada con el estado real del inventario (provisto por el **Backend de Inventario**, puerto 8000) para determinar si existe riesgo de quiebre de stock.

**Variables de negocio consideradas:**

- `stock_actual`: unidades disponibles en almacén.
- `lead_time_dias`: tiempo de entrega del proveedor (días hábiles).
- `cantidad_minima_compra`: mínimo de unidades por orden de compra.

**Fórmula del Punto de Reorden:**

```
Punto_Reorden = Promedio_Diario × Lead_Time
```

In [ ]:
# Parámetros del producto (vendrían del backend de inventario en producción)
stock_actual           = 550
lead_time_dias         = 15
cantidad_minima_compra = 10

# Cálculos logísticos
demanda_total_proyectada = int(prediccion_futura['yhat_ajustado'].sum())
promedio_diario          = prediccion_futura['yhat_ajustado'].mean()
punto_reorden_critico    = int(promedio_diario * lead_time_dias)

print(f'📦 Stock Actual en Almacén         : {stock_actual} unidades')
print(f'📈 Demanda Total Proyectada (30 d)  : {demanda_total_proyectada} unidades')
print(f'⚠️  Punto de Reorden Requerido      : {punto_reorden_critico} unidades')
print()
print('=== EVALUACIÓN LOGÍSTICA ===')

if stock_actual < demanda_total_proyectada:
    cantidad_a_comprar = max(
        cantidad_minima_compra,
        (demanda_total_proyectada - stock_actual) + punto_reorden_critico
    )
    print('🚨 STATUS: ALERTA CRÍTICA — RIESGO DE QUIEBRE DE STOCK')
    print(f'👉 Sugerencia del Sistema: Generar Orden de Compra urgente por {cantidad_a_comprar} unidades.')
    print('📧 Acción Automatizada   : Disparar notificación al Proveedor.')
else:
    print('🟢 STATUS: INVENTARIO OPTIMIZADO — Niveles seguros para el mes en curso.')

## Paso 7 — Visualización del Escenario Comercial

La gráfica siguiente representa lo que el componente `ResultChart.tsx` del Frontend renderiza mediante **Recharts** o **Plotly**.

Se comparan dos curvas:
- **Predicción Base (Inercial)**: proyección del modelo sin intervención del usuario.
- **Escenario Ajustado**: proyección modificada con los factores de primavera y competencia.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(
    prediccion_futura['ds'], prediccion_futura['yhat'],
    label='Predicción Base (Inercial)',
    color='#9ca3af', linestyle='--', linewidth=1.8
)
ax.plot(
    prediccion_futura['ds'], prediccion_futura['yhat_ajustado'],
    label='Escenario Ajustado (Primavera +25 % / Competencia −10 %)',
    color='#dc2626', linewidth=2.5
)

# Línea de stock actual
ax.axhline(
    y=stock_actual / 30,
    color='#16a34a', linestyle=':', linewidth=1.5,
    label=f'Capacidad de cobertura diaria (stock={stock_actual} u.)'
)

ax.set_title(
    'Simulador de Demanda — SINCROCIP (Cera Estampada de Abeja)',
    fontsize=14, fontweight='bold'
)
ax.set_xlabel('Fecha de Proyección', fontsize=11)
ax.set_ylabel('Unidades Demandadas / Día', fontsize=11)
ax.grid(True, linestyle=':', alpha=0.5)
ax.legend(fontsize=9, loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Paso 8 — Descomposición de Componentes del Modelo

Prophet permite visualizar de forma separada la **tendencia**, la **estacionalidad semanal** y la **estacionalidad anual** del modelo. Esta gráfica es especialmente útil para justificar ante el tribunal por qué el modelo captura el comportamiento del mercado apícola local.

In [ ]:
# Gráfica de componentes nativa de Prophet
fig_components = model_loaded.plot_components(forecast)
plt.suptitle('Descomposición del Modelo — Tendencia y Estacionalidades', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---

## Resumen para la Defensa

| Pilar | Celdas | Argumento ante el tribunal |
|---|---|---|
| **Modelado científico** | Pasos 4–5 | El sistema usa una Serie Temporal Aditiva Generalizada (Prophet), no estimaciones arbitrarias. |
| **Variables exógenas** | Paso 5 | Los factores ambientales y de mercado son configurables en tiempo real desde la UI. |
| **Automatización de decisiones** | Paso 6 | El core de valor reduce el riesgo operativo calculando el punto de reorden antes de que ocurra la pérdida económica. |
| **Transparencia del modelo** | Paso 8 | La descomposición de componentes demuestra que el modelo aprende patrones reales del negocio apícola. |

> **Archivo generado por**: Simulador de Demanda Apícola — USFX Chuquisaca  
> **Modelo utilizado**: Meta Prophet (Facebook/Meta, 2017)  
> **Lenguaje**: Python 3.10+